In [ ]:
"""
Bayesian fitting of the Mass-Metallicity Relation (MZR) using Dynesty.

Model (Equation 2):
    12 + log(O/H) = gamma_g * [log(M_star/M_sun) - 10] + Z_g10

Two modes:
    A) No outlier rejection   -> ln_likelihood_normal  (2 parameters)
    B) Outlier-robust fitting -> ln_likelihood_prune   (5 parameters)
       q_i are marginalised analytically, NOT sampled, to keep the
       problem low-dimensional and avoid the slice-sampler failure.
"""

import numpy as np
import pandas as pd
import dynesty
from dynesty import plotting as dyplot
from dynesty.utils import resample_equal
import matplotlib.pyplot as plt
from scipy.special import logsumexp


# ---------------------------------------------------------------------------
# 1.  Load data
# ---------------------------------------------------------------------------
df = pd.read_csv("MW_PEAS.csv")          # <-- replace with real path

mask = (df['STELLAR_MASS'].values>0)&(df["Z_dir_gen"]>0)&(df["Z_dir_gen_err"]>1e-5)

log_mass  = df["STELLAR_MASS"].values[mask]           # log10(M*/Msun)
mass_err  = 0.1*df["STELLAR_MASS"].values[mask]        # uncertainty on log_mass
log_OH    = df["Z_dir_gen"].values[mask]              # 12 + log(O/H)  measured
OH_err_up = 7*df["Z_dir_gen_err"].values[mask]          # upper metallicity uncertainty
OH_err_lo = 7*df["Z_dir_gen_err"].values[mask]          # lower metallicity uncertainty

N = len(log_mass)

# Number of mass draws per galaxy per likelihood call.
# Increase for more accurate marginalisation (slower); 20 is a good default.
N_MASS_DRAWS = 5


# ---------------------------------------------------------------------------
# 2.  MZR model
# ---------------------------------------------------------------------------
def model_OH(log_mass_draw, gamma_g, Z_g10):
    """Eq. 2:  predicted 12+log(O/H) given log stellar mass."""
    return gamma_g * (log_mass_draw - 10.0) + Z_g10


# ---------------------------------------------------------------------------
# 3.  Split-normal sigma (Eq. A2)
# ---------------------------------------------------------------------------
def sigma_i(OH_model, OH_truth, err_up, err_lo):
    return np.where(OH_model >= OH_truth, err_up, err_lo)


# ---------------------------------------------------------------------------
# 4.  Averaged log-likelihood over mass draws
# ---------------------------------------------------------------------------
def _mass_averaged_terms(gamma_g, Z_g10):
    mass_draws = np.random.normal(log_mass, mass_err,
                                  size=(N_MASS_DRAWS, N))
    OH_model = model_OH(mass_draws, gamma_g, Z_g10)
    sig = sigma_i(OH_model, log_OH, OH_err_up, OH_err_lo)
    sig2 = sig ** 2

    ln_inlier_draws = (
        -0.5 * np.log(2.0 * np.pi * sig2)
        - 0.5 * ((log_OH - OH_model) ** 2) / sig2
    )
    ln_inlier = logsumexp(ln_inlier_draws, axis=0) - np.log(N_MASS_DRAWS)
    mean_sig2 = np.mean(sig2, axis=0)
    return ln_inlier, mean_sig2


# ---------------------------------------------------------------------------
# 5a.  Mode A: no outlier rejection  (Eq. A1)
# ---------------------------------------------------------------------------
def ln_likelihood_normal(theta):
    gamma_g, Z_g10 = theta
    ln_inlier, _ = _mass_averaged_terms(gamma_g, Z_g10)
    return float(np.sum(ln_inlier))


# ---------------------------------------------------------------------------
# 5b.  Mode B: outlier-robust, q_i marginalised analytically (Eq. A3 + A4)
# ---------------------------------------------------------------------------
def ln_likelihood_prune(theta):
    """
    q_i marginalised analytically over [0,1] with a flat prior.

    For each galaxy:
        A_i = ln p(data_i | inlier)  + ln(1 - P_b)
        B_i = ln p(data_i | outlier) + ln(P_b)
        ln p(data_i) = logsumexp([A_i, B_i]) - ln 2
    """
    gamma_g, Z_g10, P_b, Y_b, V_b = theta

    if P_b <= 0.0 or P_b >= 1.0 or V_b <= 0.0:
        return -np.inf

    ln_inlier, mean_sig2 = _mass_averaged_terms(gamma_g, Z_g10)

    V_eff = V_b + mean_sig2
    ln_outlier = (
        -0.5 * np.log(2.0 * np.pi * V_eff)
        - 0.5 * ((log_OH - Y_b) ** 2) / V_eff
    )

    A = ln_inlier  + np.log(1.0 - P_b)
    B = ln_outlier + np.log(P_b)
    ln_marg = logsumexp(np.stack([A, B], axis=0), axis=0) - np.log(2.0)

    return float(np.sum(ln_marg))


# ---------------------------------------------------------------------------
# 6.  Prior transforms
# ---------------------------------------------------------------------------
def prior_transform_normal(u):
    return np.array([3.0 * u[0], 6.0 + 3.0 * u[1]])


def prior_transform_prune(u):
    epsilon = 1e-5
    gamma_g = 1.0 * u[0]
    Z_g10   = 6.0 + 5.0 * u[1]
    P_b     = (1.0 - 2.0 * epsilon) * u[2] + epsilon
    Y_b     = 6.0 + 3.0 * u[3]
    # V_b     = (4.0 - epsilon) * u[4] + epsilon
    log_Vb_min = -3.0
    log_Vb_max = np.log10(4.0)
    V_b = 10**((log_Vb_max - log_Vb_min) * u[4] + log_Vb_min)
    return np.array([gamma_g, Z_g10, P_b, Y_b, V_b])


# ---------------------------------------------------------------------------
# 7.  Run Nested Sampling
# ---------------------------------------------------------------------------
OUTLIER_MODE = True

if not OUTLIER_MODE:
    ndim, loglike, prior_transform = 2, ln_likelihood_normal, prior_transform_normal
    param_labels = [r"$\gamma_g$", r"$Z_{g,10}$"]
else:
    ndim, loglike, prior_transform = 5, ln_likelihood_prune, prior_transform_prune
    param_labels = [r"$\gamma_g$", r"$Z_{g,10}$", r"$P_b$", r"$Y_b$", r"$V_b$"]

sampler = dynesty.NestedSampler(
    loglike,
    prior_transform,
    ndim,
    nlive=1000,
    sample="rslice",
)
sampler.run_nested(dlogz=0.5, print_progress=True)
results = sampler.results


# ---------------------------------------------------------------------------
# 8.  Extract best-fit parameters
# ---------------------------------------------------------------------------
weights = np.exp(results.logwt - results.logz[-1])
samples = resample_equal(results.samples, weights)

for i, label in enumerate(param_labels):
    lo, med, hi = np.percentile(samples[:, i], [16, 50, 84])
    print(f"{label:>12s}  =  {med:.3f}  +{hi-med:.3f} / -{med-lo:.3f}")


# ---------------------------------------------------------------------------
# 9.  Infer per-galaxy outlier probabilities (Mode B only)
# ---------------------------------------------------------------------------
if OUTLIER_MODE:
    gamma_g_best, Z_g10_best, P_b_best, Y_b_best, V_b_best = np.median(samples, axis=0)

    ln_inlier, mean_sig2 = _mass_averaged_terms(gamma_g_best, Z_g10_best)
    V_eff = V_b_best + mean_sig2
    ln_outlier = (
        -0.5 * np.log(2.0 * np.pi * V_eff)
        - 0.5 * ((log_OH - Y_b_best) ** 2) / V_eff
    )
    A = ln_inlier  + np.log(1.0 - P_b_best)
    B = ln_outlier + np.log(P_b_best)
    log_norm = logsumexp(np.stack([A, B], axis=0), axis=0)
    q_best   = np.exp(A - log_norm)

    df["q_inlier"] = q_best
    print("\nGalaxies with q_inlier < 0.5 (likely outliers):")
    print(df[df["q_inlier"] < 0.5][["logmstellar", "Z_dir_gen", "q_inlier"]])


# ---------------------------------------------------------------------------
# 10.  Diagnostic plots
# ---------------------------------------------------------------------------
fig, axes = dyplot.cornerplot(results, labels=param_labels, show_titles=True,
                               title_kwargs={"fontsize": 11})
plt.suptitle("MZR posterior — Dynesty", y=1.02)
plt.tight_layout()
plt.savefig("mzr_corner.pdf", bbox_inches="tight")
plt.show()

13716it [01:19, 172.97it/s, bound: 34 | nc: 64 | ncall: 553113 | eff(%):  2.480 | loglstar:   -inf < -412.128 <    inf | logz: -426.023 +/-  0.112 | dlogz:  5.110 >  0.500] 


RuntimeError: Slice sampler has failed to find a valid point. Some useful output quantities:
u: [0.41774013 0.37342482 0.90699665 0.66000154 0.01296598]
nstep_left: -2.5e-323
nstep_right: 3.5e-323
nstep_hat: 6e-323
u_prop: [0.41774013 0.37342482 0.90699665 0.66000154 0.01296598]
loglstar: -412.1284073853049
logl_prop: <dynesty.utils.LoglOutput object at 0x120d556d0>
direction: [ 0.00386806 -0.01519626 -0.00148112 -0.00288088 -0.00268262]
